# 4.3 Affinity propagation

Affinity propagation turns unlabeled data into structure by choosing the right notion of similarity, compression, or surprise.

Similarity scores become messages: each point negotiates with every other point about who should represent whom. Preference and damping control how many exemplars survive and how stable those negotiations are.

Save a copy to Drive to edit. This notebook is deterministic, CPU-only, and uses only bundled scikit-learn data.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

from collections import deque
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import AffinityPropagation
from sklearn.cluster import Birch
from sklearn.cluster import DBSCAN
from sklearn.cluster import KMeans
from sklearn.cluster import MeanShift
from sklearn.cluster import OPTICS
from sklearn.cluster import estimate_bandwidth
from sklearn.datasets import load_digits
from sklearn.datasets import load_iris
from sklearn.datasets import make_blobs
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import pairwise_distances
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

SEED = 7
rng = np.random.default_rng(SEED)

def cluster_ladder():
    """D1..D5 clustering ladder of rising difficulty. Returns [(name, X, y_true, k), ...].

    y_true is the generating label (for ARI scoring only — clustering does not see it).
    Rungs: hand points -> clean blobs -> anisotropic/overlap -> real Iris -> real digits(4-class).
    """
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.3, 0.2], [3.0, 3.0], [3.2, 2.8], [0.1, 3.1], [0.2, 2.9]])
    y1 = np.array([0, 0, 1, 1, 2, 2])
    rungs.append(("D1 hand 3 clusters", x1, y1, 3))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.7, random_state=1)
    rungs.append(("D2 clean blobs", x2, y2, 3))

    x3, y3 = make_blobs(n_samples=240, centers=3, cluster_std=1.6, random_state=2)
    transform = np.array([[0.6, -0.6], [-0.4, 0.8]])
    x3 = x3 @ transform
    rungs.append(("D3 anisotropic + overlap", x3, y3, 3))

    iris = load_iris()
    rungs.append(("D4 Iris (real, 4-D)", iris.data, iris.target, 3))

    digits = load_digits()
    keep = np.isin(digits.target, [0, 1, 2, 3])
    rungs.append(("D5 digits 0-3 (real, 64-D)", digits.data[keep] / 16.0, digits.target[keep], 4))

    return rungs



def project_2d(X):
    X = np.asarray(X, dtype=float)
    if X.shape[1] == 2:
        return X
    return PCA(n_components=2, random_state=SEED).fit_transform(X)


def ari_score(y_true, labels):
    return float(adjusted_rand_score(y_true, labels))


def safe_silhouette(X, labels):
    labels = np.asarray(labels)
    keep = labels != -1
    unique = np.unique(labels[keep])
    if keep.sum() < 3:
        return np.nan
    if unique.size < 2:
        return np.nan
    if unique.size >= keep.sum():
        return np.nan
    return float(silhouette_score(X[keep], labels[keep]))


def preview_ladder(rungs):
    rows = []
    for idx, item in enumerate(rungs, start=1):
        name, X, y_true, k = item
        rows.append((idx, name, X.shape, int(np.unique(y_true).size), k, np.round(X[:3], 3).tolist()))
    return rows


def plot_cluster_panels(results, title, show_centers=False):
    fig, axes = plt.subplots(1, len(results), figsize=(18, 3.4))
    for ax, result in zip(axes, results):
        Z = project_2d(result["X"])
        ax.scatter(Z[:, 0], Z[:, 1], c=result["labels"], s=16, cmap="tab10", alpha=0.82)
        if show_centers and result.get("centers") is not None:
            centers = np.asarray(result["centers"])
            if len(centers) > 0:
                centers_2d = centers if centers.shape[1] == 2 else PCA(n_components=2, random_state=SEED).fit(result["X"]).transform(centers)
                ax.scatter(centers_2d[:, 0], centers_2d[:, 1], c="black", marker="x", s=70)
        ax.set_title(f"D{result['rung']} ARI={result['ari']:.2f}")
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(title)
    plt.show()


def plot_ari_curve(results, title):
    xs = [result["rung"] for result in results]
    ys = [result["ari"] for result in results]
    plt.figure(figsize=(6, 3.5))
    plt.plot(xs, ys, marker="o")
    plt.ylim(-0.05, 1.05)
    plt.xlabel("D1 to D5 ladder rung")
    plt.ylabel("Adjusted Rand Index")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.show()


## 3. The concept, built once on D1

The lesson normalizes local influence with $$p_i=rac{\exp(-d_i^2/2h^2)}{\sum_j\exp(-d_j^2/2h^2)}$$. Affinity propagation uses pairwise similarities instead of preset $k$, but the same scale sensitivity appears in the similarity table.

In [ ]:
# Formula audit: $p_i = \exp(-d_i^2 / 2h^2) / \sum_j \exp(-d_j^2 / 2h^2)$
h = 1.0
distances = np.array([1.0, 0.0, 1.0])
similarities = np.exp(-(distances ** 2) / (2 * h ** 2))
rounded_similarities = np.round(similarities, 3).tolist()
similarity_sum = float(np.round(similarities.sum(), 3))
central_influence = float(np.round(similarities[1] / similarities.sum(), 3))
assert rounded_similarities == [0.607, 1.0, 0.607]
assert similarity_sum == 2.213
assert central_influence == 0.452
print(np.round(similarities, 3), similarity_sum, central_influence)

Now implement the real responsibility and availability updates. Exemplars are points where responsibility plus availability is positive on the diagonal.

In [ ]:
def similarity_matrix(X):
    return -pairwise_distances(X, metric="sqeuclidean")


def method(X, k=None, preference=None, damping=0.7, max_iter=80):
    X = np.asarray(X, dtype=float)
    S = similarity_matrix(X)
    if preference is None:
        preference = float(np.median(S))
    np.fill_diagonal(S, preference)
    R = np.zeros_like(S)
    A = np.zeros_like(S)
    for _ in range(max_iter):
        AS = A + S
        max_idx = np.argmax(AS, axis=1)
        max_val = AS[np.arange(len(X)), max_idx]
        AS_second = AS.copy()
        AS_second[np.arange(len(X)), max_idx] = -np.inf
        second_val = np.max(AS_second, axis=1)
        R_new = S - max_val[:, None]
        R_new[np.arange(len(X)), max_idx] = S[np.arange(len(X)), max_idx] - second_val
        R = damping * R + (1 - damping) * R_new
        Rp = np.maximum(R, 0)
        Rp[np.diag_indices_from(Rp)] = R[np.diag_indices_from(R)]
        A_new = np.minimum(0, R.diagonal()[None, :] + Rp.sum(axis=0)[None, :] - Rp)
        A_new[np.diag_indices_from(A_new)] = Rp.sum(axis=0) - Rp.diagonal()
        A = damping * A + (1 - damping) * A_new
    exemplar_mask = np.diag(A + R) > 0
    if not np.any(exemplar_mask):
        exemplar_mask[np.argmax(np.diag(A + R))] = True
    exemplars = np.where(exemplar_mask)[0]
    labels = np.argmax(S[:, exemplars], axis=1)
    centers = X[exemplars]
    return labels, centers, exemplars

## 4. The dataset ladder

The same clustering function runs on D1 hand points, D2 clean blobs, D3 anisotropic overlap, D4 real Iris, and D5 real digits. Hidden labels are kept only for ARI scoring; the clustering method never receives them.

In [ ]:
rungs = cluster_ladder()
for row in preview_ladder(rungs):
    print(row)

## 5. Run the same method across D1-D5

The metric is Adjusted Rand Index against hidden generating labels. The labels are never passed into `method`; they are used only after clustering for evaluation.

In [ ]:
results = []
for rung, (name, X, y_true, k) in enumerate(rungs, start=1):
    X_use = StandardScaler().fit_transform(X)
    labels, centers, exemplars = method(X_use, damping=0.75)
    score = ari_score(y_true, labels)
    sil = safe_silhouette(X_use, labels)
    results.append({"rung": rung, "name": name, "X": X_use, "labels": labels, "centers": centers, "ari": score, "silhouette": sil, "exemplars": exemplars})
    print(f"D{rung} {name}: ARI={score:.3f} silhouette={sil:.3f} exemplars={len(exemplars)}")

## 6. Results visualization

Panels color each point by exemplar assignment. Black marks show exemplar locations when projected to two dimensions.

In [ ]:
plot_cluster_panels(results, "affinity propagation exemplar assignments", show_centers=True)
plot_ari_curve(results, "affinity propagation ARI vs ladder rung")

## 7. Pitfall on D5: preference and damping instability

Affinity propagation optimizes its own message criterion. On digits, small preference or damping changes can flip the number of exemplars, so we sweep both and look for stable ARI and exemplar counts.

In [ ]:
name, X5, y5, k5 = rungs[-1]
X5s = StandardScaler().fit_transform(X5)
S5 = similarity_matrix(X5s)
for multiplier in [1.5, 1.0, 0.5]:
    preference = float(np.median(S5) * multiplier)
    labels, centers, exemplars = method(X5s, preference=preference, damping=0.55)
    print("preference multiplier", multiplier, "exemplars", len(exemplars), "ARI", round(ari_score(y5, labels), 3))
for damping in [0.55, 0.75, 0.9]:
    labels, centers, exemplars = method(X5s, preference=float(np.median(S5)), damping=damping)
    print("damping", damping, "exemplars", len(exemplars), "ARI", round(ari_score(y5, labels), 3))

## 8. Evaluate it + practice

- Metric: Adjusted Rand Index vs hidden labels, plus a no-skill sanity check from random labels.
- Sanity check: rerun with a nearby seed or hyperparameter and confirm the D5 story does not flip.
- Ablation: turn off the key idea in this lesson and watch ARI or stability drop.
- Failure signal: one cluster, all noise, or many singleton clusters usually means scale or hyperparameters dominate.
- D5 check: digits are real 64-D images, so visualization is a projection while clustering uses the full feature matrix.

Ablation: set every diagonal preference very low and explain why exemplar count collapses.

Practice prompts:
1. Change one hyperparameter on D3 and explain whether ARI or the assignment panel changes first.

2. Add a stability rerun on D5 and compare the resulting ARI values.

3. Replace StandardScaler with no scaling on D4 or D5 and describe the failure mode.